<a href="https://colab.research.google.com/github/genaiconference/Agentic_KAG_Workshop_DHS_2026/blob/main/06_personalized_movie_recommendation_graphiti_living_graph_concept.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Graphiti Living Graph — Personalized Movie Recommendation Agent
## using Neo4j GraphRAG Hybrid Retrieval

---

## 1. Capstone Overview

This notebook is a **capstone demo** of an **agentic personalized movie recommendation system** built around a **Graphiti-inspired *living memory graph***.

Unlike a simple recommender that maps a query to a list of movies, this system **continuously learns from the conversation**. It updates the user's preferences in real time, **invalidates outdated preferences** (without deleting history), retrieves movies from a **Neo4j Movie Intelligence Graph** using **GraphRAG hybrid vector retrieval**, enriches missing metadata via **web search**, and produces personalized recommendations grounded in the *latest committed state* of memory.

### The system combines four building blocks

| Component | Role |
|-----------|------|
| **Graphiti-style user memory graph** | Bitemporal, living store of *who the user is* and *how their taste evolves* |
| **Neo4j Movie Intelligence Graph** | The domain knowledge base of movies, genres, themes, actors, attributes |
| **Neo4j GraphRAG hybrid vector retriever** | Retrieval engine combining semantic vector + keyword + graph filtering |
| **Agent tools** | `search_memory`, `update_memory`, `movie_hybrid_retriever`, `web_search` |

### Key concepts — know the difference

| Concept | What it is | Limitation it solves |
|---------|-----------|----------------------|
| **Static graph** | Fixed knowledge graph (e.g. the movie KG) | Great for domain facts, but doesn't evolve with the user |
| **Vector memory** | Embeddings of past messages retrieved by similarity | Remembers *text*, but has no structure, no time, no contradiction handling |
| **GraphRAG** | Retrieval combining vector search + graph traversal | Rich retrieval, but by itself is stateless about the user |
| **Living graph** | Bitemporal graph that **adds, supersedes, and invalidates** facts over time | Captures **evolving preferences with full audit history** |

### Why a *living* graph matters for personalization
- People's tastes **change**: "I used to hate horror, now I love it."
- A living graph records **when** a preference was true (`valid_from` → `valid_to`) and **why** it changed (`invalidated_by`).
- Recommendations always reflect the **current committed state**, while the full history remains **inspectable and auditable**.
- This avoids **dirty reads** — we never recommend from stale, mid-update memory.


## 2. Architecture

The agent sits between the **user** and two graphs: a **living user-memory graph** and the **Neo4j Movie Intelligence Graph**. It orchestrates four tools to produce a personalized recommendation.

```mermaid
flowchart TD
    U[User Message] --> A[🤖 Recommendation Agent]

    A -->|1. always first| SM[search_memory tool]
    A -->|2. if new signal| UM[update_memory tool]
    A -->|3. on request| MR[movie_hybrid_retriever tool]
    A -->|4. if metadata missing| WS[web_search tool]

    SM <--> LMG[(🧠 Graphiti Living<br/>User Memory Graph)]
    UM --> LMG
    MR <--> MIG[(🎞️ Neo4j Movie<br/>Intelligence Graph)]
    WS <--> NET[(🌐 External Web)]

    A --> REC[✅ Personalized<br/>Recommendation]
```

### Tool responsibilities

| Tool | Responsibility |
|------|----------------|
| **`search_memory`** | Retrieves the user's **current active** and **relevant historical** preferences from the living graph. Called **before every response**. |
| **`update_memory`** | Extracts preference signals from the latest message, creates an **Episode**, adds new facts, and **invalidates contradicting** old facts (bitemporal supersession). |
| **`movie_hybrid_retriever`** | Retrieves candidate movies from Neo4j using **hybrid** retrieval: semantic vector + keyword full-text + Cypher graph filtering. Applies likes as positive signals and dislikes/constraints as filters. |
| **`web_search`** | Validates **external / missing** movie attributes — pace, violence, freshness, popularity, reviews — used only when the graph lacks enough metadata. |

### Agent flow

```text
User Message
   ↓
search_memory                       ← always first, avoid dirty reads
   ↓
extract preference signals
   ↓
if new signal → update_memory       ← commit before recommending
   ↓
if user asks for a recommendation:
    enough context?  → No → ask ONE short clarifying question
                     → Yes → movie_hybrid_retriever
                             → web_search (only if metadata missing)
                             → validate candidates vs active dislikes
                             → recommend with reasons
else:
    acknowledge naturally (do NOT force a recommendation)
```


## 3. Environment Setup — Clone & Install

> 🛠️ **Run once (Colab-friendly).** The two cells below clone the workshop repo and install its dependencies. On a local machine you can skip the clone and install straight from your local `requirements.txt`.

| Step | Cell | Purpose |
|------|------|---------|
| 1️⃣ | `git clone …` | Pull the workshop code & data |
| 2️⃣ | `pip install -r requirements.txt` | Install Neo4j GraphRAG, Graphiti, LangChain, Tavily, etc. |


In [1]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

fatal: destination path 'Agentic_KAG_Workshop_DHS_2026' already exists and is not an empty directory.


In [2]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

## 4. Configure Credentials & Import Dependencies

Run the install cell once. This notebook requires **Neo4j** (hosting both the Movie Intelligence Graph with the GraphRAG vector + full-text indexes, and the living-memory graph), an **OpenAI** API key (used for extraction/embeddings, by the ReAct agent, and by GraphRAG), and a **Tavily** web-search key. Replace all placeholder credentials with your own before running.

> 🔑 **Tip:** store secrets in a `.env` file (loaded via `python-dotenv`) rather than hard-coding them in the notebook.


In [3]:
import os

os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Sandbox ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

# --- OpenAI ---
os.environ.setdefault(
    'OPENAI_API_KEY',
    os.getenv('OPENAI_API_KEY')
)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') # Define OPENAI_API_KEY as a Python variable

print('NEO4J_URI :', NEO4J_URI)
print('OPENAI key set:', bool(os.environ.get('OPENAI_API_KEY')))

NEO4J_URI : bolt://52.91.27.115
OPENAI key set: True


In [4]:
# ---------------------------------------------------------------------------
# Core imports
# ---------------------------------------------------------------------------
import os
import re
import json
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from langchain_core.tools import tool
import pandas as pd
from IPython.display import display, Markdown


# Web search integration (Tavily).
WEB_SEARCH_API_KEY = os.getenv("WEB_SEARCH_API_KEY")

def utc_now() -> datetime:
    """Single source of truth for timestamps (bitemporal recorded_at)."""
    return datetime.now(timezone.utc)

print("Config loaded. ")


Config loaded. 


## 5. Connect to the Neo4j Movie Intelligence Graph

We connect using the official Neo4j Python driver. The system requires an **existing** Movie Intelligence Graph with nodes like `Movie`, `Genre`, `Person`, `ProductionCompany`, `Country`, `Language`, `Keyword`, `CastMember`, and relationships such as `(:Movie)-[:HAS_GENRE]->(:Genre)`, `(:Movie)-[:TAGGED_WITH]->(:Keyword)`, `(:Movie)-[:SPOKEN_IN]->(:Language)`, `(:Movie)-[:DIRECTED_BY]->(:Person)`, `(:Person)-[:CAST_IN]->(:Movie)`.

> **Note on themes & attributes:** the graph has no dedicated `Theme` or `Attribute` nodes. Descriptive tags (e.g. *"time travel"*, *"heist"*, *"slow-burn"*, *"violent"*) are modelled as **`Keyword`** nodes via `TAGGED_WITH`, so we treat keywords as both themes and attributes for preference matching.

The catalogue lives entirely in Neo4j — the retriever queries it live, so we only run a quick count here to confirm the database is reachable and populated before running.


In [5]:
# ---------------------------------------------------------------------------
# Connect to the Neo4j Movie Intelligence Graph (required).
# ---------------------------------------------------------------------------
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
neo4j_driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI}")


# ---------------------------------------------------------------------------
# Create a movie-only vector index and a movie-only full-text index for hybrid retrieval.
# ---------------------------------------------------------------------------

from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index

create_vector_index(
    neo4j_driver,
    name="movie_only_embedding_index",
    label="Movie",
    embedding_property="movieEmbedding",
    dimensions=1536,
    similarity_fn="cosine"
)

create_fulltext_index(
    neo4j_driver,
    name="movie_only_text_index",
    label="Movie",
    node_properties=[
        "title",
        "overview",
        "tagline"
    ]
)

# Quick sanity check that the Movie Intelligence Graph is populated.
with neo4j_driver.session(database=NEO4J_DATABASE) as s:
    _movie_count = s.run("MATCH (m:Movie) RETURN count(m) AS n").single()["n"]
print(f"🎞️  Movie Intelligence Graph ready: {_movie_count} movies in Neo4j")


✅ Connected to Neo4j at bolt://52.91.27.115
🎞️  Movie Intelligence Graph ready: 496 movies in Neo4j


## 6. The Living Memory Model — powered by **Graphiti**

We initialise the [`graphiti-core`](https://github.com/getzep/graphiti) client as the temporally-aware memory backbone. Graphiti is a **temporally-aware knowledge-graph memory** with an LLM + embedder pipeline:

- **Bitemporal model** — every fact edge records `valid_at` (when it became true) and `created_at` (when we learned it) — the same bitemporal idea our living graph builds on.
- **Automatic invalidation** — when a new message contradicts an earlier fact, the old edge is stamped `invalid_at` / `expired_at` instead of being deleted, so the **full history is preserved and auditable**.
- **Hybrid semantic + keyword + graph search** over the memory.
- **Per-user namespacing** via `group_id`, so one Neo4j instance can hold many users' living graphs alongside the Movie Intelligence Graph.

The cell below wires up the OpenAI-backed LLM + embedder and defines a small **`PreferenceType` vocabulary**. This enum is used to **structure** current facts into positive/negative signals for the downstream GraphRAG movie retriever.

> 🧩 **Design note:** Sections 5–7 layer an explicit, inspectable bitemporal store and rule-based extraction *on top of* Neo4j so the workshop can trace exactly how the living graph evolves. Graphiti provides the schema, indices, and embeddings that make this possible.


In [6]:
# ---------------------------------------------------------------------------
# Initialize the real Graphiti living-memory client (OpenAI-backed).
# Graphiti does entity/fact extraction, bitemporal tracking, and contradiction
# invalidation for us — we just feed it the user's messages as episodes.
# ---------------------------------------------------------------------------
import asyncio
import nest_asyncio
from enum import Enum

from graphiti_core import Graphiti
from graphiti_core.llm_client.config import LLMConfig
from graphiti_core.embedder.openai import OpenAIEmbedder
from graphiti_core.llm_client.openai_generic_client import OpenAIGenericClient
from graphiti_core.nodes import EpisodeType

# --- LLM Model ---
OPENAI_MODEL = 'gpt-4.1-mini'

# Allow asyncio.run-style calls inside the already-running notebook event loop.
nest_asyncio.apply()

def run_async(coro):
    """Run an async Graphiti coroutine from sync notebook code."""
    return asyncio.get_event_loop().run_until_complete(coro)

from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig

# LLM for extraction (your existing code)
llm_config = LLMConfig(
    model=OPENAI_MODEL,
    small_model=OPENAI_MODEL,
)
custom_llm = OpenAIGenericClient(config=llm_config)

# EMBEDDER — use OpenAIEmbedderConfig, not raw kwargs
embedder_config = OpenAIEmbedderConfig(
    embedding_model="text-embedding-3-small"
)
embedder = OpenAIEmbedder(config=embedder_config)

# Pass BOTH to Graphiti
graphiti = Graphiti(
    NEO4J_URI,
    NEO4J_USERNAME,
    NEO4J_PASSWORD,
    llm_client=custom_llm,
    embedder=embedder  # required for fact-edge creation
)

run_async(graphiti.build_indices_and_constraints())
print("✅ Graphiti ready with LLM + Embedder.")


# ---------------------------------------------------------------------------
# We still keep a small preference vocabulary ENUM. It is NOT used to extract
# memory (Graphiti does that) — only to STRUCTURE Graphiti's current facts into
# positive/negative signals for the GraphRAG movie retriever downstream.
# ---------------------------------------------------------------------------
class PreferenceType(str, Enum):
    LIKE_GENRE          = "like_genre"
    DISLIKE_GENRE       = "dislike_genre"
    LIKE_MOVIE          = "like_movie"
    DISLIKE_MOVIE       = "dislike_movie"
    LIKE_THEME          = "like_theme"
    DISLIKE_THEME       = "dislike_theme"
    LIKE_ATTRIBUTE      = "like_attribute"
    DISLIKE_ATTRIBUTE   = "dislike_attribute"
    PREFERRED_MOOD      = "preferred_mood"
    CONTENT_CONSTRAINT  = "content_constraint"
    PACE_PREFERENCE     = "pace_preference"
    LANGUAGE_PREFERENCE = "language_preference"
    RECENCY_PREFERENCE  = "recency_preference"

print("✅ PreferenceType vocabulary ready (for retrieval signals).")

✅ Graphiti ready with LLM + Embedder.
✅ PreferenceType vocabulary ready (for retrieval signals).


## 7. Living Memory Layer — Bitemporal Preference Store in Neo4j

This section implements the **living memory graph** directly in Neo4j. Instead of deleting old preferences, every fact edge carries **bitemporal timestamps**, so we can always answer *"what is true now?"* **and** *"what was true then?"*.

```
   User message ──▶ extract preferences ──▶ write fact edge ──▶ invalidate contradictions
                                                   │
                                                   ▼
                        (:User)──[:LIKES {valid_at, invalid_at, active}]──▶(:Genre)
```

### Bitemporal fields on every edge

| Field | Meaning |
|-------|---------|
| `valid_at` | When the preference became true |
| `invalid_at` / `expired_at` | When it was superseded (`NULL` = still active) |
| `active` | Fast boolean flag for "current" edges |
| `source_text` | The original message that produced the fact (audit) |

### 🔄 Invalidation, not deletion

> If the user says *"I used to hate horror but now I love it"*, we **create** a new `LIKES → Horror` edge and **stamp** the old `DISLIKES → Horror` edge with `invalid_at`. Querying **current** facts returns only live edges, while the **full history stays inspectable** for the audit trail.

The building blocks below are:

| Function | Role |
|----------|------|
| `extract_preferences_from_text` | Rule-based extraction of likes / dislikes / avoids / watched signals |
| `write_preference_edge` | Creates a new bitemporal fact edge |
| `invalidate_conflicting_edges` | Closes out contradicting prior edges |
| `manual_add_message` / `manual_search_memory` | Public update & retrieval API |


## 8. Preference Extraction & Bitemporal Writes

The cell below is the **engine** of the living memory. It turns free-text messages into structured, timestamped facts and keeps the graph internally consistent.

### Pipeline at a glance

```
"I love comedy but I hate horror now"
            │
            ▼  extract_preferences_from_text()   (regex + clause splitting)
   ┌───────────────────────────────┐
   │ LIKES    → Genre: comedy       │
   │ DISLIKES → Genre: horror       │
   └───────────────────────────────┘
            │
            ▼  write_preference_edge()
   (:User)-[:LIKES]->(:Genre {comedy})
   (:User)-[:DISLIKES]->(:Genre {horror})
            │
            ▼  invalidate_conflicting_edges()
   any prior LIKES→horror gets invalid_at / expired_at stamped
```

### What each piece does

| Component | Responsibility |
|-----------|----------------|
| **Normalization maps** (`GENRE_NORMALIZATION`, `KEYWORD_NORMALIZATION`) | Collapse synonyms (`sci fi` → `sci-fi`, `time-travel` → `time travel`) |
| **`extract_preferences_from_text`** | Matches intent patterns (like / dislike / prefer / avoid / watched) and splits compound sentences |
| **`CONTRADICTORY_EDGE_MAP`** | Declares which edge types conflict (e.g. `LIKES` ↔ `DISLIKES`) |
| **`write_preference_edge`** | Writes the new bitemporal fact edge |
| **`manual_add_message` / `manual_search_memory`** | Public API used by the agent tools |

> 💡 **Why rule-based here?** It's deterministic, transparent, and free — ideal for a workshop demo where you want to *see* exactly which facts were extracted and why.


In [7]:
import re
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
from IPython.display import display, Markdown

# ---------------------------------------------------------------------------
# Vocabulary / normalization
# ---------------------------------------------------------------------------

CANONICAL_GENRES = {
    "action", "adventure", "animation", "comedy", "crime", "drama",
    "fantasy", "horror", "mystery", "romance", "sci-fi", "science fiction",
    "thriller", "psychological thriller", "war", "western", "documentary"
}

GENRE_NORMALIZATION = {
    "science fiction": "sci-fi",
    "scifi": "sci-fi",
    "sci fi": "sci-fi",
    "psychological thriller movie": "psychological thriller",
    "thriller movie": "thriller",
    "comedy movies": "comedy",
    "horror movies": "horror",
    "action movies": "action",
}

KEYWORD_NORMALIZATION = {
    "time travel movies": "time travel",
    "time-travel": "time travel",
    "slow paced": "slow paced",
    "slow-paced": "slow paced",
    "slow burn": "slow burn",
    "slow-burn": "slow burn",
    "mind bending": "mind-bending",
    "mind bending movies": "mind-bending",
    "children getting killed": "child harm",
    "children are harmed": "child harm",
    "child death": "child harm",
    "gore": "gore",
    "violence": "violence",
    "emotional movies": "emotional",
    "funny movies": "funny",
    "heist movies": "heist",
}


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def normalize_text(x: str) -> str:
    x = (x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_value(raw: str) -> str:
    v = normalize_text(raw).strip(" .,!?:;")
    v = re.sub(r"\s+(now|lately|these days|nowadays)$", "", v)
    v = GENRE_NORMALIZATION.get(v, v)
    v = KEYWORD_NORMALIZATION.get(v, v)
    return v


def classify_target(value: str) -> str:
    v = normalize_value(value)
    if v in CANONICAL_GENRES:
        return "Genre"
    return "Keyword"


def edge_for_polarity(intent: str) -> str:
    return {
        "like": "LIKES",
        "dislike": "DISLIKES",
        "prefer": "PREFERS",
        "avoid": "AVOIDS",
        "watched": "WATCHED",
        "want_watch": "WANTS_TO_WATCH",
    }[intent]


# ---------------------------------------------------------------------------
# Rule-based extraction (robust to clause boundaries + third-person paraphrase)
# ---------------------------------------------------------------------------
def dedupe_facts(facts: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for f in facts:
        key = (f["edge_type"], f["target_type"], f["value"])
        if key not in seen:
            seen.add(key)
            out.append(f)
    return out


def extract_preferences_from_text(text: str) -> List[Dict[str, Any]]:
    text_l = normalize_text(text)
    facts: List[Dict[str, Any]] = []

    # Stop capture at a clause boundary or common trailing connector words
    STOP = r"(?=\s+(?:and|but|,|\.|!|\?|because|since|so|for|too)\b|$)"

    patterns = [
        # ---------------- LIKE (first person) ----------------
        (r"\bi(?:'m| am)? (?:really |absolutely |totally )?(?:started |begun )?(?:liking|loving|enjoying) (.+?)" + STOP, "like"),
        (r"\bi(?:'ve| have)? (?:started|begun) to (?:like|love|enjoy) (.+?)" + STOP, "like"),
        (r"\bi really love (.+?)" + STOP, "like"),
        (r"\bi love (.+?)" + STOP, "like"),
        (r"\bi like (.+?)" + STOP, "like"),
        (r"\bi enjoy (.+?)" + STOP, "like"),
        (r"\bi adore (.+?)" + STOP, "like"),
        (r"\bi'm (?:a )?(?:big |huge |massive )?fan of (.+?)" + STOP, "like"),
        (r"\bi'm (?:really |super |very )?into (.+?)" + STOP, "like"),
        (r"\bi am (?:really |super |very )?into (.+?)" + STOP, "like"),
        (r"\bi'm obsessed with (.+?)" + STOP, "like"),
        (r"\bi dig (.+?)" + STOP, "like"),
        (r"\bmy favorite (?:genre|movie|movies|thing)? is (.+?)" + STOP, "like"),

        # ---------------- DISLIKE (first person) ----------------
        (r"\bi(?:'m| am)? (?:started |begun )?(?:disliking|hating) (.+?)" + STOP, "dislike"),
        (r"\bi(?:'ve| have)? (?:started|begun) to (?:dislike|hate) (.+?)" + STOP, "dislike"),
        (r"\bi hate (.+?)" + STOP, "dislike"),
        (r"\bi dislike (.+?)" + STOP, "dislike"),
        (r"\bi despise (.+?)" + STOP, "dislike"),
        (r"\bi can'?t stand (.+?)" + STOP, "dislike"),
        (r"\bi don'?t like (.+?)" + STOP, "dislike"),
        (r"\bi do not like (.+?)" + STOP, "dislike"),
        (r"\bi'm not (?:really |a fan of |into )(.+?)" + STOP, "dislike"),
        (r"\bi am not (?:really |a fan of |into )(.+?)" + STOP, "dislike"),
        (r"\bi'm not a big fan of (.+?)" + STOP, "dislike"),
        (r"(.+?) (?:isn'?t|is not) (?:really )?for me" + STOP, "dislike"),
        (r"\bno thanks to (.+?)" + STOP, "dislike"),

        # ---------------- PREFER / WANT (current request) ----------------
        (r"\bi prefer (.+?)" + STOP, "prefer"),
        (r"\bi'd rather watch (.+?)" + STOP, "prefer"),
        (r"\bi would rather watch (.+?)" + STOP, "prefer"),
        (r"\brecommend me (.+?)" + STOP, "prefer"),
        (r"\brecommend (?:a|an|some) (.+?)" + STOP, "prefer"),
        (r"\bsuggest (?:a|an|some)? ?(.+?)" + STOP, "prefer"),
        (r"\bgive me (.+?)" + STOP, "prefer"),
        (r"\bi feel like watching (.+?)" + STOP, "prefer"),
        (r"\bi'm in the mood for (.+?)" + STOP, "prefer"),
        (r"\bi am in the mood for (.+?)" + STOP, "prefer"),
        (r"\bi want (.+?)" + STOP, "prefer"),

        # ---------------- AVOID ----------------
        (r"\bavoid (.+?)" + STOP, "avoid"),
        (r"\babsolutely no (.+?)" + STOP, "avoid"),
        (r"\bplease no (.+?)" + STOP, "avoid"),
        (r"\bno (.+?) please" + STOP, "avoid"),
        (r"\bstay away from (.+?)" + STOP, "avoid"),
        (r"\bnothing with (.+?)" + STOP, "avoid"),
        (r"\bkeep (.+?) away" + STOP, "avoid"),
        (r"\bwithout any (.+?)" + STOP, "avoid"),

        # ---------------- WATCHED (past) ----------------
        (r"\bi(?:'ve| have)? (?:just )?(?:finished )?watched (.+?)" + STOP, "watched"),
        (r"\bi(?:'ve| have) seen (.+?)" + STOP, "watched"),
        (r"\bi saw (.+?)" + STOP, "watched"),
        (r"\bi just finished (.+?)" + STOP, "watched"),

        # ---------------- WANT TO WATCH ----------------
        (r"\bi want to watch (.+?)" + STOP, "want_watch"),
        (r"\bi'd like to watch (.+?)" + STOP, "want_watch"),
        (r"\bi would like to watch (.+?)" + STOP, "want_watch"),
        (r"\bi plan(?:ning)? to watch (.+?)" + STOP, "want_watch"),
        (r"\badd (.+?) to my watchlist" + STOP, "want_watch"),

        # ---------------- Third-person fallback (agent paraphrase safety net) ----------------
        (r"\bthe user (?:really |absolutely )?loves (.+?)" + STOP, "like"),
        (r"\bthe user (?:really |absolutely )?likes (.+?)" + STOP, "like"),
        (r"\bthe user enjoys (.+?)" + STOP, "like"),
        (r"\bthe user is (?:a fan of|into|obsessed with) (.+?)" + STOP, "like"),
        (r"\bthe user prefers (.+?)" + STOP, "prefer"),
        (r"\bthe user wants (.+?)" + STOP, "prefer"),
        (r"\bthe user (?:really |absolutely )?hates (.+?)" + STOP, "dislike"),
        (r"\bthe user dislikes (.+?)" + STOP, "dislike"),
        (r"\bthe user (?:is not|isn'?t) (?:a fan of|into) (.+?)" + STOP, "dislike"),
        (r"\bthe user (?:wants to |wants )?avoid(?:s)? (.+?)" + STOP, "avoid"),
        (r"\bthe user (?:has )?watched (.+?)" + STOP, "watched"),
        (r"\bthe user (?:has )?seen (.+?)" + STOP, "watched"),
        (r"\bthe user wants to watch (.+?)" + STOP, "want_watch"),
    ]

    for pat, intent in patterns:
        for m in re.finditer(pat, text_l):
            raw = m.group(1).strip(" .,!?:;")
            if not raw:
                continue
            value = normalize_value(raw)
            if not value:
                continue
            target_type = "Movie" if intent in {"watched", "want_watch"} else classify_target(value)
            facts.append({
                "intent": intent,
                "edge_type": edge_for_polarity(intent),
                "target_type": target_type,
                "value": value,
                "source_text": text,
            })

    # Recursively split compound sentences to catch multiple clauses
    # (e.g., "I love comedy but hate horror" or "I like action, and I avoid gore")
    for sep in [r"\bbut\b", r"\band\b"]:
        if re.search(sep, text_l) and len(facts) <= 1:
            parts = [p.strip() for p in re.split(sep, text_l) if p.strip()]
            if len(parts) > 1:
                subfacts = []
                for p in parts:
                    subfacts.extend(extract_preferences_from_text(p))
                if subfacts:
                    return dedupe_facts(subfacts)

    return dedupe_facts(facts)


# ---------------------------------------------------------------------------
# Contradiction map
# ---------------------------------------------------------------------------

CONTRADICTORY_EDGE_MAP = {
    "LIKES": ["DISLIKES", "AVOIDS"],
    "DISLIKES": ["LIKES", "PREFERS"],
    "PREFERS": ["DISLIKES", "AVOIDS"],
    "AVOIDS": ["LIKES", "PREFERS"],
    "WATCHED": [],
    "WANTS_TO_WATCH": [],
}


# ---------------------------------------------------------------------------
# Schema setup (run once)
# ---------------------------------------------------------------------------

def ensure_manual_memory_schema(neo4j_driver, NEO4J_DATABASE):
    queries = [
        "CREATE CONSTRAINT user_user_id IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE",
        "CREATE CONSTRAINT movie_name IF NOT EXISTS FOR (m:Movie) REQUIRE m.name IS UNIQUE",
        "CREATE CONSTRAINT genre_name IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE",
        "CREATE CONSTRAINT keyword_name IF NOT EXISTS FOR (k:Keyword) REQUIRE k.name IS UNIQUE",
        "CREATE CONSTRAINT person_name IF NOT EXISTS FOR (p:Person) REQUIRE p.name IS UNIQUE",
    ]
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        for q in queries:
            s.run(q)


# ---------------------------------------------------------------------------
# Invalidation + writes
# ---------------------------------------------------------------------------

def invalidate_conflicting_edges(neo4j_driver, NEO4J_DATABASE, user_id: str, edge_type: str, target_type: str, value: str) -> int:
    conflicting = CONTRADICTORY_EDGE_MAP.get(edge_type, [])
    if not conflicting:
        return 0

    cypher = f"""
    MATCH (u:User {{user_id: $user_id}})-[r]->(t:{target_type} {{name: $value}})
    WHERE type(r) IN $conflicting
      AND r.invalid_at IS NULL
      AND r.expired_at IS NULL
    SET r.invalid_at = $now,
        r.expired_at = $now,
        r.active = false
    RETURN count(r) AS closed
    """

    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            cypher,
            user_id=user_id,
            value=value,
            conflicting=conflicting,
            now=utc_now()
        ).single()
        return int(rec["closed"] or 0)


def write_preference_edge(neo4j_driver, NEO4J_DATABASE, user_id: str, fact: Dict[str, Any], conversation_id: Optional[str] = None) -> Dict[str, Any]:
    edge_type = fact["edge_type"]
    target_type = fact["target_type"]
    value = fact["value"]
    now = utc_now()

    closed = invalidate_conflicting_edges(neo4j_driver, NEO4J_DATABASE, user_id, edge_type, target_type, value)

    cypher = f"""
    MERGE (u:User {{user_id: $user_id}})
      ON CREATE SET u.name = $user_id, u.created_at = $now
    MERGE (t:{target_type} {{name: $value}})
      ON CREATE SET t.created_at = $now
    CREATE (u)-[r:{edge_type} {{
        preference_id: $preference_id,
        user_id: $user_id,
        category: $target_type,
        polarity: $polarity,
        value: $value,
        source_text: $source_text,
        conversation_id: $conversation_id,
        valid_at: $now,
        created_at: $now,
        invalid_at: null,
        expired_at: null,
        active: true
    }}]->(t)
    RETURN type(r) AS rel_type, t.name AS target
    """

    polarity = "positive" if edge_type in {"LIKES", "PREFERS", "WATCHED", "WANTS_TO_WATCH"} else "negative"

    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            cypher,
            user_id=user_id,
            value=value,
            now=now,
            preference_id=str(uuid.uuid4()),
            polarity=polarity,
            source_text=fact["source_text"],
            conversation_id=conversation_id or "default_conversation",
            target_type=target_type,
        ).single()

    return {
        "edge_type": rec["rel_type"],
        "target": rec["target"],
        "invalidated_prior_count": closed,
    }


# ---------------------------------------------------------------------------
# Public update / search API
# ---------------------------------------------------------------------------

def manual_add_message(neo4j_driver, NEO4J_DATABASE, user_id: str, message: str,
                        graphiti=None, run_async=None, EpisodeType=None) -> Dict[str, Any]:

    # Always create the Episodic node first — this replicates Graphiti's
    # episode storage and works independently of LLM extraction.
    episode = create_episodic_node(neo4j_driver, NEO4J_DATABASE, user_id, message)

    facts = extract_preferences_from_text(message)

    if not facts:
        return {
            "updated": False,
            "summary": "No structured preference facts extracted from message.",
            "facts": [],
            "episode": episode,
        }

    committed = []
    for fact in facts:
        result = write_preference_edge(neo4j_driver, NEO4J_DATABASE, user_id, fact)
        link_edge_to_episode(
            neo4j_driver, NEO4J_DATABASE, user_id,
            result["edge_type"], fact["target_type"], fact["value"],
            episode["episode_uuid"],
        )
        result["episode_uuid"] = episode["episode_uuid"]
        committed.append(result)

    return {
        "updated": True,
        "summary": f"Committed {len(committed)} explicit preference fact(s).",
        "facts": committed,
        "episode": episode,
    }


def manual_search_memory(neo4j_driver, NEO4J_DATABASE, user_id: str, query: Optional[str] = None, current_only: bool = True) -> Dict[str, Any]:
    cypher = """
    MATCH (u:User {user_id: $user_id})-[r]->(t)
    WHERE type(r) IN ['LIKES','DISLIKES','PREFERS','AVOIDS','WATCHED','WANTS_TO_WATCH']
    """
    if current_only:
        cypher += """
        AND r.invalid_at IS NULL
        AND r.expired_at IS NULL
        """
    cypher += """
    RETURN type(r) AS edge_type,
           labels(t)[0] AS target_type,
           t.name AS target,
           r.source_text AS source_text,
           r.valid_at AS valid_at,
           r.invalid_at AS invalid_at
    ORDER BY r.valid_at DESC
    """

    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rows = [dict(x) for x in s.run(cypher, user_id=user_id)]

    facts = [f"{r['edge_type']} {r['target_type']}:{r['target']}" for r in rows]

    return {
        "active_preferences": facts,
        "preference_summary": "; ".join(facts) if facts else "No known preferences yet (cold start).",
        "rows": rows,
    }


def inspect_manual_memory_graph(neo4j_driver, NEO4J_DATABASE, user_id: str, current_only: bool = True) -> pd.DataFrame:
    cypher = """
    MATCH (u:User {user_id: $user_id})-[r]->(t)
    WHERE type(r) IN ['LIKES','DISLIKES','PREFERS','AVOIDS','WATCHED','WANTS_TO_WATCH']
    """
    if current_only:
        cypher += """
        AND r.invalid_at IS NULL
        AND r.expired_at IS NULL
        """
    cypher += """
    RETURN u.user_id AS user_id,
           type(r) AS rel_type,
           labels(t)[0] AS target_type,
           t.name AS target,
           r.valid_at AS valid_at,
           r.invalid_at AS invalid_at,
           r.source_text AS source_text
    ORDER BY r.valid_at DESC
    """

    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rows = [dict(x) for x in s.run(cypher, user_id=user_id)]
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Smoke test (uncomment after ensure_manual_memory_schema has been run once)
# ---------------------------------------------------------------------------

# ensure_manual_memory_schema(neo4j_driver, NEO4J_DATABASE)
# uid = "user_harika"
# print(manual_add_message(neo4j_driver, NEO4J_DATABASE, uid, "I love horror movies and prefer to watch them"))
# print(manual_add_message(neo4j_driver, NEO4J_DATABASE, uid, "The user absolutely loves horror movies."))
# print(manual_search_memory(neo4j_driver, NEO4J_DATABASE, uid)["preference_summary"])
# inspect_manual_memory_graph(neo4j_driver, NEO4J_DATABASE, uid)

## 9. Episodic Memory — Recording *What Was Said*

Preferences answer *"what does the user like?"*. **Episodes** answer *"where did that belief come from?"*. Each raw user message is stored as an `Episodic` node and linked to the entities it mentioned — mirroring how Graphiti stores episodes.

```
(:User)-[:HAS_EPISODE]->(:Episodic {content:"I love heist movies"})
                              │
                              └─[:MENTIONS]->(:Keyword {heist})
```

| Function | Role |
|----------|------|
| `ensure_episodic_schema` | Uniqueness constraint on `Episodic.uuid` |
| `create_episodic_node` | Stores the verbatim message with a timestamp |
| `link_edge_to_episode` | Connects a fact edge back to its source episode (`MENTIONS`) |
| `inspect_episodes` | Human-readable view of the episode timeline |

> 🧾 **Provenance for free:** every preference edge can be traced back to the exact sentence that created it — essential for explainability and auditing.


In [42]:
# ---------------------------------------------------------------------------
# Manual Episodic node creation (replicates Graphiti's episode storage)
# ---------------------------------------------------------------------------

def ensure_episodic_schema(neo4j_driver, NEO4J_DATABASE):
    # Graphiti may already have created a plain (non-unique) index on
    # Episodic.uuid. A uniqueness constraint cannot be created while such an
    # index exists, so we tolerate that case — the uuid is generated with
    # uuid4() and is unique in practice regardless.
    cypher = "CREATE CONSTRAINT episodic_uuid IF NOT EXISTS FOR (e:Episodic) REQUIRE e.uuid IS UNIQUE"
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        try:
            s.run(cypher)
        except Exception as e:
            # e.g. Neo.ClientError.Schema.IndexAlreadyExists when Graphiti
            # already indexed Episodic.uuid — safe to ignore.
            print(f"ℹ️ Skipping Episodic uuid constraint (already indexed): {e}")




def create_episodic_node(neo4j_driver, NEO4J_DATABASE, user_id: str, message: str,
                          source: str = "message",
                          source_description: str = "user chat message") -> Dict[str, Any]:
    """Manually create an Episodic node, mirroring Graphiti's own schema."""
    now = utc_now()
    episode_uuid = str(uuid.uuid4())
    episode_name = f"msg-{now.timestamp()}"

    cypher = """
    MERGE (u:User {user_id: $user_id})
      ON CREATE SET u.name = $user_id, u.created_at = $now
    CREATE (e:Episodic {
        uuid: $uuid,
        name: $name,
        group_id: $user_id,
        content: $content,
        source: $source,
        source_description: $source_description,
        valid_at: $now,
        created_at: $now
    })
    CREATE (u)-[:HAS_EPISODE {created_at: $now}]->(e)
    RETURN e.uuid AS uuid, e.name AS name
    """

    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            cypher,
            user_id=user_id,
            uuid=episode_uuid,
            name=episode_name,
            content=message,
            source=source,
            source_description=source_description,
            now=now,
        ).single()

    return {"episode_uuid": rec["uuid"], "episode_name": rec["name"]}


def link_edge_to_episode(neo4j_driver, NEO4J_DATABASE, user_id: str,
                          edge_type: str, target_type: str, value: str,
                          episode_uuid: str) -> None:
    """Link the just-created preference edge to its source Episodic node,
    and connect the Episodic node to the mentioned entity (MENTIONS),
    the same way Graphiti links episodes to nodes."""
    cypher = f"""
    MATCH (u:User {{user_id: $user_id}})-[r:{edge_type}]->(t:{target_type} {{name: $value}})
    WHERE r.invalid_at IS NULL AND r.expired_at IS NULL
    WITH r, t
    ORDER BY r.created_at DESC
    LIMIT 1
    MATCH (e:Episodic {{uuid: $episode_uuid}})
    SET r.episode_uuid = $episode_uuid
    MERGE (e)-[:MENTIONS]->(t)
    """
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        s.run(
            cypher,
            user_id=user_id,
            value=value,
            episode_uuid=episode_uuid,
        )

In [9]:
def inspect_episodes(neo4j_driver, NEO4J_DATABASE, user_id: str) -> pd.DataFrame:
    cypher = """
    MATCH (u:User {user_id: $user_id})-[:HAS_EPISODE]->(e:Episodic)
    OPTIONAL MATCH (e)-[:MENTIONS]->(t)
    RETURN e.uuid AS episode_uuid,
           e.name AS episode_name,
           e.content AS content,
           e.created_at AS created_at,
           collect(DISTINCT t.name) AS mentioned_entities
    ORDER BY e.created_at DESC
    """
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rows = [dict(x) for x in s.run(cypher, user_id=user_id)]
    return pd.DataFrame(rows)

## 10. Neo4j GraphRAG Hybrid Movie Retriever Tool

`movie_hybrid_retriever` wraps **Neo4j GraphRAG's `HybridRetriever`**, which fuses two complementary retrieval signals:

```
                 ┌─────────────────────┐
  user query ───▶│  🔎 semantic vector  │──┐
                 │  (embedding search)  │  │   Reciprocal Rank
                 └─────────────────────┘  ├──▶  Fusion ──▶ top-k movies
                 ┌─────────────────────┐  │
  user query ───▶│  🔤 full-text search  │──┘
                 │  (keyword / BM25)    │
                 └─────────────────────┘
```

It builds the query from the user's **active** preferences: likes become **positive signals** that steer the search, while dislikes and content constraints become **hard filters** applied to the results.

| Step | What happens |
|------|--------------|
| 1. `preferences_to_signals` | Split memory into likes / dislikes / constraints |
| 2. `HybridRetriever.search` | Vector + full-text hybrid recall of candidate titles |
| 3. `enrich_movies` | Graph expansion — genres, keywords, cast, directors, languages |
| 4. Filtering | Drop any movie matching an active dislike or content-risk constraint |

**Required indexes** (created earlier): `movie_only_embedding_index` (vector) and `movie_only_text_index` (full-text).

Each candidate returns: `title`, `genres`, `keywords`, `directors`, `cast`, `overview`, `rating`, `release_date`, `why_retrieved`, `matched_preferences`, `possible_risks`.


In [10]:
def enrich_movies(driver, titles):
    query = """
    MATCH (m:Movie)
    WHERE m.title IN $titles

    OPTIONAL MATCH (m)-[:HAS_GENRE]->(g:Genre)
    OPTIONAL MATCH (m)-[:TAGGED_WITH]->(k:Keyword)
    OPTIONAL MATCH (m)-[:DIRECTED_BY]->(d:Person)
    OPTIONAL MATCH (p:Person)-[:CAST_IN]->(m)
    OPTIONAL MATCH (m)-[:PRODUCED_BY]->(pc:ProductionCompany)
    OPTIONAL MATCH (m)-[:PRODUCED_IN]->(c:Country)
    OPTIONAL MATCH (m)-[:SPOKEN_IN]->(l:Language)

    RETURN
        m.title AS title,
        m.overview AS overview,
        m.vote_average AS rating,

        collect(DISTINCT g.name) AS genres,
        collect(DISTINCT k.name) AS keywords,
        collect(DISTINCT d.name) AS directors,
        collect(DISTINCT p.name) AS cast,
        collect(DISTINCT pc.name) AS production_companies,
        collect(DISTINCT c.name) AS countries,
        collect(DISTINCT l.name) AS languages
    """

    return driver.execute_query(
        query,
        titles=titles,
        database_="neo4j"
    ).records

In [11]:
# ---------------------------------------------------------------------------
# Helper: turn structured signals (from facts_to_signals) into positive
# signals + negative filters for retrieval.
# ---------------------------------------------------------------------------
def preferences_to_signals(signal_dicts: List[Dict[str, Any]]) -> Dict[str, Any]:
    likes_genre, likes_theme, likes_attr, moods = [], [], [], []
    dislikes, constraints = [], []
    for p in signal_dicts:
        pt, obj = p["preference_type"], p["object"].lower()
        if pt == "like_genre":         likes_genre.append(obj)
        elif pt == "like_theme":       likes_theme.append(obj)
        elif pt == "like_attribute":   likes_attr.append(obj)
        elif pt == "preferred_mood":   moods.append(obj)
        elif pt in ("dislike_genre", "dislike_theme", "dislike_attribute", "dislike_movie"):
            dislikes.append(obj)
        elif pt == "content_constraint":
            constraints.append(obj)
    return {"likes_genre": likes_genre, "likes_theme": likes_theme,
            "likes_attr": likes_attr, "moods": moods,
            "dislikes": dislikes, "constraints": constraints}


# Map content constraints to attribute red-flags for filtering.
CONSTRAINT_RISK = {
    "avoid child harm": ["violent", "gory", "dark"],
    "avoid gore": ["gory"],
}


# ---------------------------------------------------------------------------
# Neo4j GraphRAG hybrid retriever (vector index + full-text index).
# Requires pre-built indexes `movie_overview_vector` and `movie_fulltext`.
# ---------------------------------------------------------------------------

from neo4j_graphrag.retrievers import HybridRetriever
from neo4j_graphrag.embeddings import OpenAIEmbeddings
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
neo4j_driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI}")

_embedder = OpenAIEmbeddings(model="text-embedding-3-small")

_hybrid_retriever = HybridRetriever(
    driver=neo4j_driver,
    vector_index_name="movie_only_embedding_index",
    fulltext_index_name="movie_only_text_index",
    embedder=_embedder,
)


def _graphrag_retrieve(query_text: str, top_k: int) -> List[Dict[str, Any]]:
    """
    Returns lightweight movie candidates from HybridRetriever.
    """
    import ast

    res = _hybrid_retriever.search(query_text=query_text, top_k=top_k)

    out = []

    for item in res.items:

        rec = {}

        if item.content:
            try:
                rec.update(ast.literal_eval(item.content))
            except Exception:
                pass

        rec.update(item.metadata or {})

        out.append(rec)

    return out


@tool
def movie_hybrid_retriever(user_id: str, recommendation_query: str = "",
                           current_preferences: Optional[List[Dict[str, Any]]] = None,
                           top_k: int = 4) -> List[Dict[str, Any]]:
    """Retrieve candidate movies from the Neo4j Movie Intelligence Graph via
    GraphRAG hybrid vector + graph retrieval, honoring likes (positive signals)
    and dislikes / content constraints (filters). Never returns movies that
    violate active dislikes or content constraints.
    """
    # Always ground on the LATEST committed living memory (avoid dirty reads).
    current_preferences = current_preferences or []
    signals = preferences_to_signals(current_preferences)

    query_text = recommendation_query or " ".join(
        signals["likes_theme"] + signals["likes_genre"] + signals["likes_attr"]) or "movie"

    raw = _graphrag_retrieve(query_text, top_k * 4)


    candidate_titles = [
    movie["title"]
    for movie in raw
    if movie.get("title")
]

    # Expand every movie using the graph
    movies = enrich_movies(neo4j_driver, candidate_titles)

    candidates = []

    for m in movies:
        genres = {
                        g.lower()
                        for g in m.get("genres", [])
                    }

        themes = {
                      k.lower()
                      for k in m.get("keywords", [])
                  }

        attrs = set(themes)


        # ---- hard filter: dislikes ----
        blocked = (genres & set(signals["dislikes"])) or \
                  (themes & set(signals["dislikes"])) or \
                  (attrs  & set(signals["dislikes"]))
        # ---- hard filter: content constraints ----
        risks = []
        for c in signals["constraints"]:
            flagged = attrs & set(CONSTRAINT_RISK.get(c, []))
            if flagged:
                risks.append(f"{c}: flagged by {sorted(flagged)}")
        if blocked or risks:
            continue  # exclude violating movies entirely

        matched = []

        matched += [
            f"genre:{g}"
            for g in genres & set(signals["likes_genre"])
        ]

        matched += [
            f"keyword:{k}"
            for k in themes & set(signals["likes_theme"])
        ]

        matched += [
            f"attribute:{a}"
            for a in attrs & set(signals["likes_attr"])
        ]

        candidates.append({
                            "title": m["title"],
                            "overview": m.get("overview", ""),
                            "rating": m.get("rating", 0),
                            "release_date": m.get("release_date"),

                            "genres": m.get("genres", []),
                            "keywords": m.get("keywords", []),
                            "directors": m.get("directors", []),
                            "cast": m.get("cast", []),
                            "production_companies": m.get("production_companies", []),
                            "countries": m.get("countries", []),
                            "languages": m.get("languages", []),

                            "matched_preferences": matched,

                            "why_retrieved":
                                f"Matched GraphRAG search for '{query_text}'",

                            "possible_risks": risks,
                        })
    return candidates[:top_k]


print("✅ Tool defined: movie_hybrid_retriever (Neo4j GraphRAG HybridRetriever)")

✅ Connected to Neo4j at bolt://52.91.27.115
✅ Tool defined: movie_hybrid_retriever (Neo4j GraphRAG HybridRetriever)


## 11. Web Search Tool: `web_search`

Used **only** when the graph lacks enough metadata or a candidate needs validation (e.g. *is this slow-paced? violent? recent? highly rated?*). Backed by **Tavily** — set `WEB_SEARCH_API_KEY` to your Tavily API key.

> 🌐 **Guardrail:** the agent is instructed to reach for the web *only* when the knowledge graph is missing an attribute — this keeps recommendations grounded and token-cheap.

Returns: `external_summary`, `results`, `source`.


In [12]:
# ---------------------------------------------------------------------------
# Tool 4: web_search — validates external / missing movie attributes.
# Uses Tavily. Set WEB_SEARCH_API_KEY (Tavily API key) to enable.
# ---------------------------------------------------------------------------
from tavily import TavilyClient
from langchain_core.tools import tool

_tavily = TavilyClient(api_key=WEB_SEARCH_API_KEY)

@tool
def web_search(query: str, movie_title: str = "") -> Dict[str, Any]:
    """Validate external / missing movie attributes (pace, violence, freshness,
    popularity, reviews). Use ONLY when the graph lacks enough metadata.
    Returns a short direct answer only, to minimize token usage.
    """
    q = f"{movie_title} {query}".strip()
    res = _tavily.search(query=q, max_results=1, include_answer=True)

    answer = res.get("answer", "").strip()

    if answer:
        return {"external_summary": answer, "source": "tavily"}

    # Fallback only if Tavily couldn't synthesize a direct answer
    results = res.get("results", [])
    if results:
        top = results[0]
        return {
            "external_summary": top.get("content", "")[:150],
            "source": top.get("url", "tavily"),
        }

    return {"external_summary": "No information found.", "source": "tavily"}


print("✅ Tool defined: web_search (Tavily)")


✅ Tool defined: web_search (Tavily)


## 12. ReAct Agent Prompt (tool policy)

Instead of a hand-coded control loop, we use a **LangChain ReAct agent**. The LLM itself decides — turn by turn — which tool to call, in what order, using the **ReAct** *Thought → Action → Action Input → Observation* loop.

The prompt below tells the agent **what each tool does and when to call it**, and enforces the **memory-first workflow**: search memory first, and **continuously call `update_memory` the moment the user states or changes any preference**, so the living graph is always up to date before recommending.


In [29]:
AGENT_INSTRUCTIONS = """
You are a personalized movie recommendation assistant powered by a living Movie Memory Graph.

Your primary goal is to recommend movies that best match the user's CURRENT request while using long-term memory only to personalize recommendations, enforce established preferences, and improve future conversations.

The user's current request always has higher priority than historical preferences unless it conflicts with an active hard constraint.

--------------------------------------------------
AVAILABLE TOOLS
--------------------------------------------------

searchmemory
Retrieves the latest committed user preferences and historical facts from the living memory graph.

updatememory
Creates or updates long-term user preference facts in the memory graph.

moviehybridretriever
Retrieves candidate movies using Neo4j graph retrieval combined with semantic retrieval.

websearch
Verifies movie metadata when required.
Use only when retrieval metadata is missing, ambiguous, or safety-critical.

--------------------------------------------------
TOOL EXECUTION POLICY
--------------------------------------------------

Every conversation MUST begin by calling searchmemory.

Never recommend movies before reasoning over the latest committed memory state.

Never mention tools or memory operations to the user.

If the user provides new long-term preference information, update memory before retrieving recommendations.

If the user only shares information and is not asking for a recommendation, acknowledge naturally and do not force a recommendation.

--------------------------------------------------
MEMORY MODEL
--------------------------------------------------

Treat user information as one of three categories.

1. Long-Term Preferences

These describe the user's enduring tastes or persistent constraints.

Examples:
"I love horror."
"Thrillers are my favorite."
"I never watch musicals."
"I hate excessive gore."
"I usually prefer psychological thrillers."
"I always avoid sad endings."

These SHOULD be stored.

2. Temporary Session Preferences

These apply only to the current recommendation request unless the user clearly signals that they are stable over time.

Examples:
"Recommend horror tonight."
"I feel like comedy."
"Something lighter."
"Less intense."
"Give me a fun movie."
"This time give me action."

These should guide the current recommendation flow, but should NOT be permanently stored unless the user clearly indicates they reflect an ongoing preference.

3. Feedback

Feedback about recommendations helps improve future recommendations.

Examples:
"I've already seen it."
"Didn't enjoy it."
"Too slow."
"Too depressing."
"Loved the ending."

Store the underlying reason whenever possible instead of only storing the title.
Prefer:
"Dislikes bleak movies"
instead of:
"Dislikes Movie X"
unless the title itself is meaningful.

--------------------------------------------------
WHEN TO UPDATE MEMORY
--------------------------------------------------

Update memory when the user reveals new enduring information about themselves.

Examples:
✓ I love horror.
✓ I usually watch thrillers.
✓ I hate gore.
✓ I always avoid sad endings.
✓ My favorite actor is Tom Hanks.

Do NOT update memory for:
- temporary moods
- one-off requests
- clarification answers
- short session-only narrowing replies

unless they clearly express a stable long-term preference.

Examples:
✗ Recommend comedy tonight.
✗ Something lighter.
✗ Surprise me.
✗ This time give me action.

If the user explicitly says not to remember something, never call updatememory.

Whenever possible, combine all preference updates from a single user message into one updatememory call.

--------------------------------------------------
REQUEST PRIORITY
--------------------------------------------------

Always interpret user intent in the following order:

Priority 1: The user's current request.
Priority 2: The user's clarification answer during the current recommendation flow.
Priority 3: Active hard constraints.
Priority 4: Long-term stored preferences.
Priority 5: Historical recommendations and prior session feedback.

Current intent always wins unless doing so would violate a hard content constraint.

--------------------------------------------------
HARD CONSTRAINTS
--------------------------------------------------

Hard constraints must never be violated.

Examples include:
- avoid gore
- no sexual content
- no horror
- no violence
- family-friendly only

If retrieval returns movies violating hard constraints, reject those candidates.

If satisfying the user's request is impossible without breaking a hard constraint, explain briefly and ask whether the user would like to relax one constraint.

--------------------------------------------------
RECOMMENDATION WORKFLOW
--------------------------------------------------

When the user requests recommendations, follow these steps exactly.

Step 1
Call searchmemory.

Step 2
Identify exactly what the user asked for THIS TURN.

Never silently inject old preference words into the retrieval query.

Example:
Memory: likes sci-fi
Current request: recommend comedy
Retrieve comedy.
Do NOT retrieve sci-fi comedy unless the user requested it.

Step 3
Determine whether clarification is needed.

If clarification is unnecessary, retrieve immediately.

If clarification is necessary, ask exactly ONE short clarifying question.

After the user answers, never ask another clarification question during the same recommendation attempt.

Proceed directly to retrieval.

Step 4
Update memory only if the user revealed a stable long-term preference.

Step 5
Call moviehybridretriever.

Step 6
Evaluate retrieved movies against active hard constraints.
Reject conflicting candidates.

Step 7
Use websearch only if required to verify uncertain metadata.
Never guess.

Step 8
Rank remaining candidates.

Step 9
Generate recommendations.

--------------------------------------------------
CLARIFICATION RULES
--------------------------------------------------

Ask a clarifying question only when it will materially improve the recommendation.

Do NOT ask questions simply because more information could be useful.

If the current request is already specific enough to retrieve strong recommendations, retrieve immediately.

Never ask more than ONE clarifying question during a recommendation attempt.

After the user answers that clarification, treat the answer as the final narrowing signal and immediately continue to retrieval.

Do not decompose the user's answer into additional questions.

If the user gives a usable modifier or tonal answer, retrieve immediately.

Examples that are already specific enough:
- light horror
- psychological thriller
- grounded sci-fi
- twisty mystery
- comedy with romance
- slow-burn mystery
- fun sci-fi
- darker thriller
- horror with lighter tone
- less intense thriller
- something spooky

Do NOT split usable tonal answers into finer forced choices.

Bad behavior:
User: "light horror"
Assistant: "spooky-comedy or mildly creepy?"

Bad behavior:
User: "lighter tone"
Assistant: "playful or atmospheric?"

Good behavior:
User gives a usable tonal or vibe signal.
Assistant retrieves immediately.

If the same request already contains a narrowing modifier, DO NOT ask another clarification.

--------------------------------------------------
BROAD GENRE RULE
--------------------------------------------------

Treat a single broad genre request as potentially ambiguous.

Examples:
Horror
Thriller
Comedy
Drama
Romance
Action
Sci-fi
Mystery
Fantasy
Adventure
Crime

For these requests:

If multiple substantially different interpretations would produce noticeably different recommendations, ask ONE short clarifying question.

Otherwise, retrieve directly.

Never ask a clarification simply because the request contains one genre word.

If retrieval can confidently produce excellent recommendations, retrieve immediately.

However, for broad requests like "thriller" or "horror", if the active memory or context suggests multiple different plausible directions, prefer asking one short clarifying question before retrieval.

Examples:
User: "Horror"
Good clarification: "Do you want psychological, supernatural, or something lighter?"

User: "Thriller"
Good clarification: "Psychological thriller or action thriller?"

User: "Comedy"
Good clarification: "Looking for something romantic, family-friendly, or dark?"

If the same request already contains a narrowing modifier, retrieve immediately.

Examples:
- light horror
- psychological thriller
- fun sci-fi
- grounded crime drama
- slow-burn mystery
- twisty thriller
- dark fantasy
- family comedy
- romantic comedy
- military action

--------------------------------------------------
FOLLOW-UP REPLIES
--------------------------------------------------

Treat short follow-up replies as meaningful narrowing signals for the CURRENT recommendation attempt.

Examples:
- Lighter.
- Less intense.
- More fun.
- More grounded.
- Something spooky.
- Darker.
- Recent.
- Older.
- Classic.

These are sufficient.

Do not ask another question.
Retrieve immediately.

These follow-up replies should generally be treated as temporary session guidance, not permanent memory, unless the user clearly frames them as a stable preference.

--------------------------------------------------
ALREADY WATCHED
--------------------------------------------------

If the user says:
"I've seen it."
"I already watched that."

Do NOT infer dislike.

Simply avoid recommending that title again during the current conversation.

--------------------------------------------------
REJECTED RECOMMENDATIONS
--------------------------------------------------

If the user rejects a recommendation:
"Not interested."
"No."
"Anything else?"

Treat this as feedback for the current session.

Recommend different movies.

Do not recommend previously rejected titles again in the same conversation.

Update memory only if the rejection clearly expresses a stable preference.

Example:
"I don't like depressing movies."
Store:
"Avoids depressing movies"

--------------------------------------------------
RETRIEVAL FAILURE
--------------------------------------------------

If movie retrieval returns no suitable candidates:

First, relax only soft filters.

Examples of soft filters:
- release year
- popularity
- runtime
- language

Do NOT relax:
- hard dislikes
- hard constraints

If retrieval still fails, retry once with softer non-safety filters.

If no suitable movies are found, explain honestly and ask whether the user would like to relax one constraint.

--------------------------------------------------
RANKING STRATEGY
--------------------------------------------------

Rank candidate movies using the following priority:

1. Hard constraints
2. Current user request
3. Current clarification answer
4. Long-term preferences
5. Previously recommended or already watched titles (avoid repetition)
6. Recommendation diversity
7. Overall movie quality
8. Popularity

Never sacrifice relevance for popularity.

--------------------------------------------------
MULTI-INTENT REQUESTS
--------------------------------------------------

If the user asks for recommendations AND asks an informational question in the same message, handle both.

Example:
"Recommend a horror movie and tell me why Hereditary is so popular."

Recommend movies.
Answer the informational question.
Do not ignore either part.

--------------------------------------------------
WEB SEARCH POLICY
--------------------------------------------------

Prefer information returned by moviehybridretriever.

Use websearch only when:
- metadata is missing
- metadata is ambiguous
- content safety must be verified
- recency must be confirmed

Never use websearch for information already available in retrieval results.

Never hallucinate movie facts.

If uncertain, verify before answering.

--------------------------------------------------
RESPONSE STYLE
--------------------------------------------------

Talk like a knowledgeable movie enthusiast, not a customer support bot.

Be:
- concise
- natural
- confident
- helpful

Avoid:
- corporate language
- robotic phrasing
- excessive disclaimers
- explaining internal reasoning
- mentioning tools or memory systems

--------------------------------------------------
RECOMMENDATION FORMAT
--------------------------------------------------

When recommending movies, provide 3-5 titles by default unless the user asks for a different number.

For each recommendation include:
- the movie title
- release year if known
- one short reason it matches the request
- the general tone or vibe

Avoid spoilers.

Example style:
1. Movie Title (2024) - A tense psychological thriller with strong twists and a fast pace.
2. Another Movie (2023) - Lighter supernatural horror with good humor and minimal gore.

After listing recommendations, add one short sentence naming the best starting pick.

Example:
"If I were picking one to start with tonight, I'd go with Movie Title."

--------------------------------------------------
EXPLANATION QUALITY
--------------------------------------------------

Prioritize useful explanations over generic praise.

Good:
"Matches your request for a lighter horror because it focuses more on atmosphere and mystery than on gore."

Bad:
"Great movie that many people enjoy."

--------------------------------------------------
REACT FORMAT
--------------------------------------------------

You MUST follow this format exactly.

Thought: brief reasoning
Action: searchmemory | updatememory | moviehybridretriever | websearch
Action Input: tool input
Observation: tool result

Repeat as needed.

Finish with:

Thought: final reasoning
Final Answer: user-facing response

--------------------------------------------------
CLARIFYING QUESTION FORMAT
--------------------------------------------------

A clarifying question is NOT a tool call.

Use:

Thought: brief reasoning
Final Answer: <one short clarifying question>

Never output:
- Question:
- Action: None
- empty Action Input
- multiple questions

--------------------------------------------------
EXAMPLE FLOWS
--------------------------------------------------

Example 1 - Broad genre

User: Horror

Thought: broad genre request is ambiguous
Final Answer: Do you want psychological, supernatural, or something lighter?

--------------------------------------------------
Example 2 - Narrowed request

User: Light horror

Thought: request is already specific enough
Action: moviehybridretriever
Action Input: {"user_id": "{user_id}", "recommendation_query": "light horror movies with fun spooky atmosphere and minimal gore"}
Observation: ...

Thought: final reasoning
Final Answer: ...

--------------------------------------------------
Example 3 - Temporary request should not be stored

User: Recommend comedy tonight

Thought: temporary session preference; no memory update
Action: moviehybridretriever
Action Input: {"user_id": "{user_id}", "recommendation_query": "best comedy movies for tonight"}
Observation: ...

Thought: final reasoning
Final Answer: ...

--------------------------------------------------
Example 4 - Stable preference should be stored

User: I usually avoid depressing movies

Thought: enduring preference should be stored
Action: updatememory
Action Input: {"user_id": "{user_id}", "usermessage": "I usually avoid depressing movies and sad endings."}
Observation: updated

Thought: acknowledge preference
Final Answer: Got it — I'll avoid recommending especially bleak or heavy endings.

--------------------------------------------------
FINAL SELF-CHECK
--------------------------------------------------

Before every final answer, verify all of the following:

Did I call searchmemory first?

Did I update memory only for enduring preferences?

Did I avoid storing temporary session requests and short clarification answers unless they were clearly stable preferences?

Did I ask at most ONE clarifying question?

Did the user already provide a usable modifier such as "light", "dark", "psychological", "grounded", "spooky", "fun", "lighter tone", or "less intense"? If yes, retrieve immediately.

Am I silently injecting old preferences that the user did not confirm this turn? If yes, remove them unless they are hard constraints.

Did I verify uncertain metadata before stating it?

Am I avoiding movies already rejected or already watched during this conversation?

Is my recommendation explanation specific and spoiler-free?

If all checks pass, produce the final answer.

The current user's id is: {user_id}
"""

## 13. Build the LangChain ReAct Agent

We build a **LangChain ReAct agent** with `create_react_agent`. The LLM dynamically decides which tool to call and when — there is **no hard-coded response logic**. The four tools are bound to the current `user_id` and exposed as single-input LangChain tools so the ReAct loop can call them.

The agent runs an `AgentExecutor` that loops *Thought → Action → Observation* until it produces a final answer. Set `OPENAI_API_KEY` (or the Azure variables) so the agent has an LLM to drive it.


In [30]:
# ---------------------------------------------------------------------------
# LangChain ReAct agent. The LLM decides which tool to call and when — there is
# NO hard-coded reply logic. Tools are wrapped as single-input LangChain tools
# bound to the active user_id. update_memory is called by the LLM whenever the
# user shares/changes a preference, so the living graph updates continuously.
# ---------------------------------------------------------------------------
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks import BaseCallbackHandler


# --- 1) Pick the OpenAI LLM that drives the ReAct loop. ---
def _build_llm():
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(api_key=OPENAI_API_KEY, model=OPENAI_MODEL, temperature=0.3)


# --- 2) Callback handler to capture the tool-call trace (for the chat UI). ---
class ToolTraceHandler(BaseCallbackHandler):
    def __init__(self):
        self.trace: List[str] = []
    def on_tool_start(self, serialized, input_str, **kwargs):
        name = (serialized or {}).get("name", "tool")
        self.trace.append(name)


# --- 3) Build single-input tools bound to a user_id. -------------------------
from langchain_core.tools import Tool
import json


def _make_tools_for_user(user_id: str) -> List[Tool]:

    def _t_search_memory(query: str) -> str:
        res = manual_search_memory(
            neo4j_driver, NEO4J_DATABASE,
            user_id=user_id,
            query=query or None,
            current_only=True,
        )
        return json.dumps(res, default=str)

    def _t_update_memory(user_message: str) -> str:
        res = manual_add_message(
            neo4j_driver, NEO4J_DATABASE,
            user_id=user_id,
            message=user_message,
        )
        return json.dumps(res, default=str)

    def _t_movie_hybrid_retriever(recommendation_query: str) -> str:
        mem = manual_search_memory(neo4j_driver, NEO4J_DATABASE, user_id=user_id, current_only=True)
        rows = mem.get("rows", [])

        current_preferences = []
        for r in rows:
            edge = r["edge_type"]
            value = r["target"]
            if edge == "LIKES":
                pt = "like_genre" if r["target_type"] == "Genre" else "like_theme"
            elif edge == "DISLIKES":
                pt = "dislike_genre" if r["target_type"] == "Genre" else "dislike_theme"
            elif edge == "PREFERS":
                pt = "preferred_mood"
            elif edge == "AVOIDS":
                pt = "content_constraint"
            else:
                continue
            current_preferences.append({"object": value, "preference_type": pt})

        cands = movie_hybrid_retriever.invoke({
            "user_id": user_id,
            "recommendation_query": recommendation_query,
            "current_preferences": current_preferences,
            "top_k": 4,
        })
        return json.dumps(cands, default=str)

    def _t_web_search(query: str) -> str:
        # query may be "movie title | question" split if provided that way
        title = query.split("|")[0].strip() if "|" in query else ""
        res = web_search.invoke({"query": query, "movie_title": title})
        return json.dumps(res, default=str)

    tools = [
        Tool(
            name="search_memory",
            func=_t_search_memory,
            description=(
                "Search the user's current active movie preferences. "
                "Always call this FIRST before answering."
            ),
        ),
        Tool(
            name="update_memory",
            func=_t_update_memory,
            description=(
                "Commit a new user preference to living memory. "
                "IMPORTANT: Pass the user's ORIGINAL message text VERBATIM as Action Input — "
                "do NOT paraphrase, summarize, or rewrite it in third person. "
                "Example: if the user said 'I love horror movies', "
                "Action Input must be exactly 'I love horror movies'."
            ),
        ),
        Tool(
            name="movie_hybrid_retriever",
            func=_t_movie_hybrid_retriever,
            description=(
                "Retrieve candidate movies from the Neo4j Movie Intelligence Graph "
                "based on current preferences and the recommendation query. "
                "Use this when the user asks for a recommendation."
            ),
        ),
        Tool(
            name="web_search",
            func=_t_web_search,
            description=(
                "Validate external/missing movie attributes (pace, violence, rating, "
                "freshness) or confirm whether a candidate violates an active dislike "
                "or content constraint. Input: 'movie title | question'. "
                "Use ONLY when graph metadata is incomplete or uncertain — do not guess."
            ),
        ),
    ]
    return tools

# --- 4) ReAct prompt template (standard scaffold + our instructions). --------
from langchain_core.prompts import PromptTemplate

REACT_TEMPLATE = """
Answer the following question as best you can.

{instructions}

You have access to the following tools:
{tools}

Use this exact format:

Question: the input question you must answer
Thought: think about the next step
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Rules:
- Output only the fields shown above.
- Do not add commentary outside this format.
- After each Thought, output either Action or Final Answer.
- Do not output both Action and Final Answer in the same step.
- If you use a tool, always provide Action Input.

Begin!

Question: {input}
{agent_scratchpad}
"""
prompt = PromptTemplate(
    template=REACT_TEMPLATE,
    input_variables=["input", "tools", "tool_names", "agent_scratchpad"],
)


# --- 5) Assemble the agent executor. -----------------------------------------
_LLM = _build_llm()

def build_react_agent_for_user(user_id: str) -> AgentExecutor:
    tools = _make_tools_for_user(user_id)
    prompt = PromptTemplate.from_template(REACT_TEMPLATE).partial(
        instructions=AGENT_INSTRUCTIONS.format(user_id=user_id))
    react_agent = create_react_agent(_LLM, tools, prompt)
    return AgentExecutor(agent=react_agent, tools=tools, verbose=True,
                         handle_parsing_errors=True, max_iterations=8)


def run_agent(user_id: str, user_message: str) -> Dict[str, Any]:
    """Run one dynamic ReAct turn. Returns {'response', 'trace'}."""
    executor = build_react_agent_for_user(user_id)
    handler = ToolTraceHandler()
    out = executor.invoke({"input": user_message}, config={"callbacks": [handler]})
    return {"response": out.get("output", ""),
            "trace": " → ".join(handler.trace) or "(no tools)"}


print("✅ ReAct agent ready.")


✅ ReAct agent ready.


## 14. 💬 Interactive Chat UI (free-flow conversation)

Talk to the agent live, directly inside the notebook. Type any message — share a preference (*"I love heist movies"*), change your mind (*"actually I'm into slow-burn now"*), or ask for a recommendation (*"what should I watch?"*).

Each turn is handled dynamically by the **ReAct agent**: the LLM decides which tools to call (`search_memory`, `update_memory`, `movie_hybrid_retriever`, `web_search`). The **living memory graph updates in real time** — watch the tool trace and current preferences change as you chat, and use **Show audit trail** to see invalidated history.


In [47]:
import html as _html
import ipywidgets as widgets


CHAT_USER = "user_harika"  # dedicated user_id for the interactive session


def _reset_chat_memory():
    """Clear this chat user's manual living-memory graph for a fresh session."""
    try:
        with neo4j_driver.session(database=NEO4J_DATABASE) as s:
            s.run("""
                MATCH (u:User {user_id: $uid})-[r]->(t)
                WHERE type(r) IN ['LIKES','DISLIKES','PREFERS','AVOIDS','WATCHED','WANTS_TO_WATCH']
                DELETE r
            """, uid=CHAT_USER)

            s.run("""
                MATCH (u:User {user_id: $uid})-[r:HAS_EPISODE]->(e:Episodic)
                DELETE r
            """, uid=CHAT_USER)

            s.run("""
                MATCH (e:Episodic {group_id: $uid})
                OPTIONAL MATCH (e)-[m]-()
                DELETE m, e
            """, uid=CHAT_USER)

    except Exception as e:
        print("Reset note:", e)


def inspect_graphiti_memory_graph(driver, database, user_id, current_only=True):
    cypher = """
    MATCH ()-[r:RELATES_TO]->()
    WHERE r.group_id = $user_id
    """
    if current_only:
        cypher += """
          AND r.invalid_at IS NULL
          AND r.expired_at IS NULL
        """
    cypher += """
    RETURN
        r.fact AS fact,
        r.valid_at AS valid_at,
        r.created_at AS created_at,
        r.invalid_at AS invalid_at,
        r.expired_at AS expired_at
    ORDER BY r.valid_at DESC, r.created_at DESC
    """
    with driver.session(database=database) as s:
        rows = [dict(x) for x in s.run(cypher, user_id=user_id)]
    return pd.DataFrame(rows)


def _render_transcript(history: List[Dict[str, str]]) -> str:
    css = """
    <style>
      .lg-chat{font-family:Segoe UI,Arial,sans-serif;max-width:760px}
      .lg-row{display:flex;margin:6px 0}
      .lg-user{justify-content:flex-end}
      .lg-bot{justify-content:flex-start}
      .lg-bubble{padding:10px 14px;border-radius:16px;max-width:76%;
                 white-space:pre-wrap;line-height:1.4;font-size:14px}
      .lg-user .lg-bubble{background:#2563eb;color:#fff;border-bottom-right-radius:4px}
      .lg-bot  .lg-bubble{background:#f1f5f9;color:#0f172a;border-bottom-left-radius:4px}
      .lg-meta{font-size:11px;color:#64748b;margin-top:2px}
    </style>
    """
    rows = []
    for turn_ in history:
        role, text = turn_["role"], _html.escape(turn_["text"])
        if role == "user":
            rows.append(f'<div class="lg-row lg-user"><div class="lg-bubble">{text}</div></div>')
        else:
            trace = turn_.get("trace", "")
            meta = f'<div class="lg-meta">🧾 tools: {_html.escape(trace)}</div>' if trace else ""
            rows.append(f'<div class="lg-row lg-bot"><div>'
                        f'<div class="lg-bubble">{text}</div>{meta}</div></div>')
    return css + '<div class="lg-chat">' + "".join(rows) + "</div>"


_history: List[Dict[str, str]] = []

transcript = widgets.HTML(value=_render_transcript(_history))
text_in = widgets.Text(placeholder="Type a message… e.g. 'I love heist movies' or 'recommend something'",
                       layout=widgets.Layout(width="560px"))
send_btn = widgets.Button(description="Send", button_style="primary", icon="paper-plane")
reset_btn = widgets.Button(description="Reset memory", icon="trash")
audit_btn = widgets.Button(description="Show audit trail", icon="history")
show_state = widgets.Checkbox(value=True, description="Show live memory graph")
state_out = widgets.Output()


def _refresh_state():
    state_out.clear_output(wait=True)
    if show_state.value:
        with state_out:
            display(Markdown(f"**🧠 Current active living memory for `{CHAT_USER}`**"))
            df = inspect_manual_memory_graph(neo4j_driver, NEO4J_DATABASE, CHAT_USER, current_only=True)
            display(df if not df.empty else "No active facts yet.")


def _handle_send(_=None):
    msg = text_in.value.strip()
    if not msg:
        return
    text_in.value = ""
    _history.append({"role": "user", "text": msg})
    result = run_agent(CHAT_USER, msg)
    _history.append({"role": "bot", "text": result["response"],
                     "trace": result["trace"]})
    transcript.value = _render_transcript(_history)
    _refresh_state()


def _handle_reset(_=None):
    _reset_chat_memory()
    _history.clear()
    transcript.value = _render_transcript(_history)
    _refresh_state()


def _handle_audit(_=None):
    state_out.clear_output(wait=True)
    with state_out:
        display(Markdown(f"**📜 Full audit trail for `{CHAT_USER}` (history preserved)**"))
        df = inspect_manual_memory_graph(neo4j_driver, NEO4J_DATABASE, CHAT_USER, current_only=False)
        display(df if not df.empty else "No facts yet.")


send_btn.on_click(_handle_send)
reset_btn.on_click(_handle_reset)
audit_btn.on_click(_handle_audit)
text_in.on_submit(_handle_send)

controls = widgets.HBox([text_in, send_btn, reset_btn, audit_btn])
ui = widgets.VBox([transcript, controls, show_state, state_out])
_refresh_state()
display(Markdown("### 🎬 Chat with your living-memory movie agent"))
display(ui)

### 🎬 Chat with your living-memory movie agent

## 15. Recap — What Makes This Graph *Living*

```
┌──────────────┐   search_memory   ┌─────────────────────┐
│   👤 User     │ ────────────────▶ │ 🧠 Living Memory KG  │
│  message     │ ◀──────────────── │ (bitemporal facts)  │
└──────┬───────┘   update_memory    └─────────┬───────────┘
       │                                       │ current preferences
       ▼                                       ▼
┌──────────────┐  movie_hybrid_retriever ┌─────────────────────┐
│ 🤖 ReAct Agent│ ──────────────────────▶ │ 🎞️ Movie KG (GraphRAG)│
└──────┬───────┘        web_search        └─────────────────────┘
       ▼
  ✅ Personalized, audit-safe recommendation
```

| Capability | Where it lives |
|------------|----------------|
| 🧠 **Evolving preferences** | Bitemporal edges with `valid_at` / `invalid_at` |
| 🔄 **Invalidation, not deletion** | `invalidate_conflicting_edges` |
| 🧾 **Provenance** | `Episodic` nodes + `MENTIONS` links |
| 🔎 **Grounded retrieval** | Neo4j GraphRAG `HybridRetriever` |
| 🌐 **Fresh facts on demand** | Tavily `web_search` |
| 🤖 **Autonomous orchestration** | LangChain ReAct agent |

### Try it yourself 🎬
- *"I love heist movies"* → watch a `LIKES → heist` edge appear.
- *"Actually I'm into slow-burn now"* → new fact added, old one invalidated.
- *"What should I watch?"* → grounded recommendation from the **latest** committed state.
- Click **Show audit trail** to see the full, preserved history.
